In [ ]:
!pip install datasketch

In [ ]:
!pip install langdetect

In [ ]:
import duckdb
import pandas as pd
import re
import shutil
import ast
import json
from tqdm.auto import tqdm
import random
import hashlib
from pathlib import Path
from langdetect import detect, DetectorFactory
from datasketch import MinHash, MinHashLSH

In [ ]:
file_path = 'drive/MyDrive/messages.csv'
df = pd.read_csv(file_path, low_memory=False)

is_propaganda = (df['binary_label'] == True) | (df['narrative_id'].notna()) | (df['sub_narrative_id'].notna())

df_prop = df[is_propaganda].copy()

df_base = df[~is_propaganda].sample(
    n=min(len(df) - len(df_prop), 4 * len(df_prop)),
    random_state=2301
).copy()

df_combined = pd.concat([df_prop, df_base], ignore_index=True)
df_combined.to_csv('drive/MyDrive/files_prop/messages_combined.csv', index=False)


In [ ]:
DetectorFactory.seed = 0
random.seed(42)
LSH_THRESHOLD = 0.90
NUM_PERM = 128

In [ ]:
BOILERPLATE_PATTERNS = [
    r'підписатись\s*\|\s*надіслати\s*новину',
    r'більше\s*новин\s*тут',
    r'читайте\s*також',
    r'підписуйтесь\s*на\s*(?:telegram|facebook|x|liga).*?(?:важливе|новини)?',
    r'читайте\s*нас\s*[ву]\s*telegram.*?(?:новини)?',
    r'за\s*темою',
    r'сайт\s*facebook\s*youtube',
    r'надіслати\s*новину',
    r'читати\s*більше',
    r'еспресо\s*долучайтесь',
    r'stoprussia\s*stop\s*russian\s*aggression\s*russia\s*invaded\s*ukraine',
    r'source:', r'источник:', r'follow\s*us', r'subscribe', r'read\s*more',
    r'register\s*now\s*for\s*free\s*unlimited\s*access\s*to\s*reuters\.\s*com\s*register',
    r'our standards:\s*the thomson reuters trust principles\.',
    r'new you can now listen to fox news articles!?',
    r'click here to get the fox news app',
    r'підписатись\s*\|\s*надіслати інфо про техніку',
    r'разом з еспресо!\s*підписуйтесь на telegram-канал:?',
    r'стежте за найважливішими новинами львова, регіону, україни(?:\s*та\s*світу)?',
    r'підписатися\s*підтримати',
    r'Брати по Зброї.*',
    r'ус\s*підписатися',
    r'труха\s*(?:україна|харьков|київ|харків)(?:\s*прислать\s*новость)?',
    r'україна\s*online\s*підписатись',
    r'підпишись\s*на\s*укрінформ.*',
    r'rbc\s*ua\s*ексклюзиврбкукраїна',
    r'rbc\s*ua',
    r'liga\s*net',
    r'liganet',
    r'babel',
    r'censor\s*net.*',
    r'радіо\s*свобода\s*підписуйтесь',
    r'ми\s*у\s*(?:whatsapp|viber|telegram|facebook|instagram|twitter|tik\s*tok).*',
    r'підпишіться\s*на\s*nv',
    r'типовий\s*львів',
    r'терміновий\s*репост\s*близьким',
    r'читайте\s*за\s*посиланням',
    r'подробиці\s*$',
    r'деталі\s*$',
    r'детальніше\s*$',
    r'докладніше\s*$',
    r'на\s*фото\s*:',
    r'фото\s*reuters.*',

    r'разом\s*з\s*["\']?еспресо["\']?\s*!?\s*(?:захід|захід!)?\s*-?\s*канал\s*:?',
    r'слідкуйте\s*за\s*подіями\s*в\s*україні\s*та\s*світі\s*(?:разом\s*з\s*["\']?еспресо["\']?\s*!?\s*-?\s*канал\s*:?)?',
    r'підписуйтесь\s*на\s*нашу\s*facebook[-\s]?сторінку\.?',
    r'підпишіться\s*на\s*щоденну\s*e-?mail\s*розсилку.*?підписатись\.?',
    r'-\s*канал\s*:\s*$',

    r'(?:additional\s+)?reporting\s+by\s+.*?(?:editing\s+by\s+.*)?$',
    r'writing\s+by\s+.*?(?:editing\s+by\s+.*)?$',
    r'opinions\s+expressed\s+are\s+those\s+of\s+the\s+author\.?',
    r'\(\s*[\d.,]+\s*euros?\s*\)\s*reporting\s+by.*$',

    r'sign\s*up\s*to\s*the\s*daily\s*.*?\s*email\s*or\s*follow\s*.*?\s*on\s*twitter\s*at\b',

    r'follow\s+along\s+as\s+fox\s+news.*$',
    r'camera\s+by\s+.*$',
    r'twitter\s*:\s*\.?\s*$',
    r'click\s*here\s*to\s*sign\s*up\s*for\s*(?:the|our)\s*.*?\s*newsletter',

    r'\bджерело\s*:',
    r'\bдослівно\s*:',
    r'\bпряма\s*мова\s*:',
    r'\bquote\s*:',
    r'\bdetails\s*:',
    r'\bmore\s*details\s*:',
    r'\bbackground\s*:',
    r'\bsummary\s*:',
    r'\breminder\s*:',
    r'\bpreviously\s*:',
    r'\bcorrection\s*:',
    r'\bupdate\s*:',
    r'\bнагадаємо\s*,?\s*(?:що)?',
    r'\bнагадуємо\s*,?\s*(?:що)?',
    r'\bяк\s*повідомлялося(?:\s*раніше)?\s*,?',

    # --- нові: варіанти CTA спільнот/соцмереж, не покриті раніше ---
    r'запроси(?:ти)?\s*знайомих\s*(?:і|та)?\s*друзів\s*підписатись.*',
    r'слідкуйте\s*за\s*нами\s*у\s*соціальних\s*мережах\s*:?.*$',

    # артефакти навігації фотогалерей
    r'\bnext\s+image\s+\d+\s+of\s+\d+\s+prev\b',
    r'\bfullscreen\b',
]

In [ ]:
# --- нове: видалення вбудованих заголовків "інших статей" (Fox News-стиль) ---
# Fox News (і подібні) вставляють прямо посеред тексту статті посилання на
# суміжні матеріали у вигляді СУЦІЛЬНО ВЕЛИКИХ ЛІТЕР заголовків
# (наприклад: 'RUSSIA INVADES UKRAINE: LIVE UPDATES',
# 'CLICK HERE TO SIGN UP FOR THE ENTERTAINMENT NEWSLETTER',
# 'QUEEN ELIZABETH RECEIVES INTERNATIONAL WOMEN'S DAY TRIBUTE...').
# Такі вставки не мають фіксованого тексту (кожна стаття — інша), тому їх
# не можна покрити списком літеральних патернів; натомість ловимо структурну
# ознаку — 5+ поспіль латинських слів, написаних лише великими літерами.
# Кирилиця (українські абревіатури на кшталт ЗСУ, СБУ) НЕ зачіпається, бо
# використовує інший діапазон символів (не [A-Z]).
_CAPS_HEADLINE_RE = re.compile(r"(?:[A-Z][A-Z'\-]*[\s:;,]+){4,}[A-Z][A-Z'\-]*")

def strip_embedded_caps_headlines(text: str, min_words=5):
    def _repl(m):
        words = re.findall(r"[A-Z][A-Z'\-]*", m.group(0))
        return '' if len(words) >= min_words else m.group(0)
    return _CAPS_HEADLINE_RE.sub(_repl, text)

In [ ]:
# strip_intradoc_repeated_phrases, _TOKEN_RE, _normalize_token — ВИДАЛЕНО.
# Токен-based дедуп повторів давав мало вигоди (артефакт рідкісний, лише
# у довгих лонгрідах) і ніс ризик видаляти навмисні повторювані гасла/рефрени
# пропаганди — тобто сам сигнал, який має вимірювати ентропія.
# Замість цього — простіший і безпечніший підхід: капати довжину
# документа перед аналізом, щоб рідкісні викиди-лонгріди не спотворювали
# метрику, і лишити лише консервативний sentence-level дедуп нижче.

MAX_DOC_CHARS = 4000  # символів; довші документи обрізаються після очищення

_SENTENCE_SPLIT_RE = re.compile(r'(?<=[.!?])\s+')

def strip_intradoc_repeats(text: str, min_sentence_len=15, max_sentence_len=300, min_repeats=3):
    """
    Видаляє речення, які повторюються кілька разів ВСЕРЕДИНІ одного й того ж
    тексту (на відміну від strip_generic_boilerplate_per_file, яка ловить
    повтори МІЖ різними документами файлу). Консервативний: працює лише
    на рівні речень з пунктуацією, тому ризик зачепити навмисні
    пропагандистські рефрени низький — короткий рефрен типу "Слава Україні!"
    зазвичай коротший за min_sentence_len або трапляється <3 разів.
    """
    if not isinstance(text, str) or not text.strip():
        return text

    sentences = [s.strip() for s in _SENTENCE_SPLIT_RE.split(text) if s.strip()]
    norm_counts = {}
    norm_to_first = {}

    for s in sentences:
        if not (min_sentence_len <= len(s) <= max_sentence_len):
            continue
        ns = normalize_sentence(s)
        if not ns:
            continue
        norm_counts[ns] = norm_counts.get(ns, 0) + 1
        norm_to_first.setdefault(ns, s)

    to_remove = {norm_to_first[ns] for ns, c in norm_counts.items() if c >= min_repeats}
    if not to_remove:
        return text

    for s in sorted(to_remove, key=len, reverse=True):
        text = text.replace(s, "")

    return re.sub(r'\s+', ' ', text).strip()

In [ ]:
_TOKEN_RE = re.compile(r'\S+')

def _normalize_token(tok: str):
    t = tok.lower()
    t = re.sub(r'^\W+|\W+$', '', t)
    return t

In [ ]:
TELEGRAM_HEADER_RE = re.compile(
    r'^(?::\d{2}|\d{2}:\d{2}(?::\d{2})?(?:\+\d{2}:\d{2})?|\d{4}-\d{2}-\d{2}[T\s]\d{2}:\d{2}:\d{2}(?:\+\d{2}:\d{2})?)?\s*'
    r'\d*\s*'
    r'(?:None|unknown|\d+)\s*'
    r'(?:None|unknown|\d+|\{.*?\}|\S+)\s*'
    r'(?:None|unknown|no|photo|video\S*|doc\S*|application\S*|media)\s*',
    re.IGNORECASE
)

In [ ]:
def extract_content(raw_val):
    if isinstance(raw_val, str):
        clean_str = raw_val.strip()
        if clean_str.startswith('[') or clean_str.startswith('{'):
            try:
                parsed = ast.literal_eval(raw_val)
                if isinstance(parsed, list): raw_val = parsed
            except Exception: pass
    if isinstance(raw_val, list):
        target_texts = []
        for item in raw_val:
            if isinstance(item, dict) and item.get('role') == 'user':
                content = str(item.get('content', ''))
                parts = re.split(r'Text:\s*', content, flags=re.IGNORECASE)
                if len(parts) > 1:
                    target = parts[-1]
                    target = re.split(r'\n\s*\n\s*(?:1\.|Identify the|Only return|Remember to|Standard label)', target)[0]
                    target_texts.append(target.strip(' "\'\n'))
                else:
                    target_texts.append(content)
        if target_texts: return " ".join(target_texts)
    return str(raw_val)

In [ ]:
def strip_known_boilerplate(text: str):
    for pattern in BOILERPLATE_PATTERNS:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    return text

In [ ]:
def process(text):
    text = extract_content(text)
    text = re.sub(r'[\n\r]+', '. ', text)
    text = text.replace('\t', ' ')

    text = re.sub(r'[\u200B-\u200D\uFEFF]', ' ', text)

    text = re.sub(r'(?:https?://|www\.)\S+', '', text)
    text = re.sub(r'\b\w+\.\w+/\S+', '', text)

    text = re.sub(r'@\w+\s*', '', text)
    text = re.sub(r'#(\w+)', lambda m: re.sub(r'([a-zа-яіїєґ])([A-ZА-ЯІЇЄҐ])', r'\1 \2', m.group(1)), text)
    text = re.sub(r'\d+/\d+\s*$', '', text)

    text = re.sub(r"\{'_':\s*'MessageReplyHeader'.*?\}", "", text)
    text = re.sub(r"^\{.*?\}\s*", "", text)
    text = TELEGRAM_HEADER_RE.sub("", text)
    text = re.sub(r"^(?::\d{2}|\d{2}:\d{2}(?::\d{2})?)\s+\d+\s+.*?(?=\s[А-Яа-яA-Za-z🔴⚡️🇺АК])", "", text)

    text = re.sub(r'^\W*.{0,250}?\([Rr]euters\)\s*[—–-]\s*', '', text)

    date_pattern = r'^\W*(?:.{1,100}?[—–|-]\s*)?(?:(?:Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday|Понеділок|Вівторок|Середа|Четвер|П\'ятниця|Субота|Неділя),?\s*)?(?:\d{1,2}\s+[A-Za-zА-Яа-яІіЇїЄєҐґ]+\s+\d{4}|[A-Za-zА-Яа-яІіЇїЄєҐґ]+\s+\d{1,2},?\s+\d{4})[,\s\d:]*\s*(?:[—–-]\s*)?'
    text = re.sub(date_pattern, '', text, flags=re.IGNORECASE)

    text = re.sub(r'([a-zа-яіїєґ])([A-ZА-ЯІЇЄҐ])', r'\1 \2', text)
    text = re.sub(r'([A-ZА-ЯІЇЄҐ]+)([A-ZА-ЯІЇЄҐ][a-zа-яіїєґ]+)', r'\1 \2', text)
    text = re.sub(r'([.!?])([A-Za-zА-Яа-яІіЇїЄєҐґ])', r'\1 \2', text)

    text = re.sub(r'&[a-zA-Z0-9#]+;', ' ', text)
    text = re.sub(r'<.*?>', ' ', text)

    text = strip_embedded_caps_headlines(text)
    text = strip_known_boilerplate(text)

    text = re.sub(r'_{3,}', ' ', text)

    text = strip_intradoc_repeats(text)

    text = re.sub(r'[«»“”]', '"', text)
    text = re.sub(r'[‘’]', "'", text)
    text = re.sub(r'[—–]', '-', text)

    text = re.sub(r'[^\w\s.,!?"\'\-:;()«»“”]', ' ', text)

    text = re.sub(r'\s+', ' ', text)
    text = text.strip()

    if len(text) > MAX_DOC_CHARS:
        text = text[:MAX_DOC_CHARS].rsplit(' ', 1)[0]

    return text.strip()

In [ ]:
def detect_lang(text):
    if not isinstance(text, str) or not text.strip(): return "unknown"
    try: return detect(text[:500])
    except Exception: return "unknown"

In [ ]:
def normalize_sentence(sentence: str):
    sentence = sentence.lower()
    sentence = re.sub(r'https?://\S+|www\.\S+', ' ', sentence)
    sentence = re.sub(r'\S+@\S+', ' ', sentence)
    sentence = re.sub(r'<[^>]+>', ' ', sentence)
    sentence = re.sub(r'[«»„“”"\'`]', '', sentence)
    sentence = re.sub(r'[-–—]', ' ', sentence)
    sentence = re.sub(r'[^\w\s]', ' ', sentence)
    sentence = re.sub(r'\s+', ' ', sentence)
    return sentence.strip()

In [ ]:
_SENTENCE_SPLIT_RE = re.compile(r'(?<=[.!?])\s+')

In [ ]:
def strip_generic_boilerplate_per_file(texts: list, min_count=5, min_ratio=0.25, min_sentence_len=15):
    if not texts: return texts, []
    unique_texts = list(dict.fromkeys(texts))
    n = len(unique_texts)
    if n < min_count: return texts, []

    avg_text_len = sum(len(t) for t in unique_texts) / n if n > 0 else 200
    effective_max_len = min(200, 0.5 * avg_text_len)

    norm_counts = {}
    norm_to_raw = {}
    position_ratios = {}
    for text in unique_texts:
        sentences = {s.strip() for s in _SENTENCE_SPLIT_RE.split(text) if min_sentence_len <= len(s.strip()) <= effective_max_len}
        for s in sentences:
            norm_s = normalize_sentence(s)
            if not norm_s: continue
            norm_counts[norm_s] = norm_counts.get(norm_s, 0) + 1
            norm_to_raw.setdefault(norm_s, set()).add(s)

            idx = text.find(s)
            if idx != -1 and len(text) > 0:
                position_ratios.setdefault(norm_s, []).append(idx / len(text))

    threshold = max(min_count, int(min_ratio * n))

    def is_edge_positioned(norm_s):
        ratios = position_ratios.get(norm_s, [])
        if not ratios: return False
        edge_hits = sum(1 for r in ratios if r <= 0.2 or r >= 0.8)
        return edge_hits / len(ratios) >= 0.6

    boilerplate_norms = {ns for ns, c in norm_counts.items() if c >= threshold and is_edge_positioned(ns)}

    if not boilerplate_norms: return texts, []

    sentences_to_remove = set()
    for ns in boilerplate_norms: sentences_to_remove.update(norm_to_raw[ns])
    sentences_to_remove_sorted = sorted(list(sentences_to_remove), key=len, reverse=True)

    cleaned = []
    for text in texts:
        for s in sentences_to_remove_sorted: text = text.replace(s, "")
        cleaned.append(re.sub(r"\s+", " ", text).strip())
    return cleaned, sentences_to_remove_sorted


In [ ]:
def alpha_ratio_fast(text):
    if not text: return 0
    return len(re.findall(r'[^\W\d_]', text)) / len(text)

In [ ]:
def get_minhash(text, num_perm=128):
    m = MinHash(num_perm=num_perm)
    tokens = text.split()
    if len(tokens) < 3:
        for t in tokens: m.update(t.encode('utf8'))
    else:
        for i in range(len(tokens) - 2):
            m.update(f"{tokens[i]} {tokens[i+1]} {tokens[i+2]}".encode('utf8'))
    return m

In [ ]:
def find_text_column(df):
    candidates = ['text', 'text_clean', 'full_text', 'tweet', 'tweet_text', 'message', 'messages', 'content', 'article', 'body', 'post']
    for col in df.columns:
        if str(col).lower() in candidates: return col
    text_cols = [col for col in df.columns if df[col].dtype == 'object']
    if not text_cols: return None
    best_col, max_len = None, -1
    for col in text_cols:
        avg_len = df[col].astype(str).str.len().mean()
        if avg_len > max_len: max_len, best_col = avg_len, col
    return best_col

In [ ]:
con = duckdb.connect()
base_dir = Path('drive/MyDrive/files_prop')

In [ ]:
skip_keywords = ['mapping', 'meta']
supported_ext = {'.parquet', '.csv', '.tsv', '.jsonl', '.txt'}
archive_ext = {'.zip', '.tar', '.gz', '.rar', '.7z'}
system_files = {'.ds_store', 'thumbs.db'}

In [ ]:
files_to_process = []
for f in base_dir.rglob('*'):
    if not f.is_file(): continue
    name_lower = f.name.lower()
    ext_lower = f.suffix.lower()
    if name_lower in system_files or name_lower.startswith('._'): continue
    if ext_lower in archive_ext or ext_lower not in supported_ext: continue
    if any(kw in name_lower for kw in skip_keywords): continue
    files_to_process.append(f)

def get_priority(file_path):
    name = file_path.name.lower()
    if 'narrative' in name or 'hierarchy' in name: return 1
    if 'labeled' in name or 'hiqualprop' in name or 'hqp' in name: return 2
    return 3

files_to_process.sort(key=lambda f: (get_priority(f), str(f)))

In [ ]:
global_lsh = MinHashLSH(threshold=LSH_THRESHOLD, num_perm=NUM_PERM)
global_exact_hashes = {}
global_id_counter = 0

In [ ]:
save_path_jsonl = 'drive/MyDrive/clean_propaganda_dataset.jsonl'
save_path_parquet = 'drive/MyDrive/clean_propaganda_dataset.parquet'

In [ ]:
local_jsonl = '/content/clean_propaganda_dataset.jsonl'
local_parquet = '/content/clean_propaganda_dataset.parquet'

drive_jsonl = 'drive/MyDrive/clean_propaganda_dataset.jsonl'
drive_parquet = 'drive/MyDrive/clean_propaganda_dataset.parquet'

print("Processing...")
with open(local_jsonl, 'w', encoding='utf-8') as out_jsonl:
    for file_path in tqdm(files_to_process, desc="Processing files"):
        ext = file_path.suffix.lower()
        try:
            if ext == '.parquet': df = con.execute(f"SELECT * FROM read_parquet('{file_path}')").df()
            elif ext == '.csv': df = pd.read_csv(file_path, on_bad_lines='skip', low_memory=False)
            elif ext == '.tsv': df = pd.read_csv(file_path, sep='\t', on_bad_lines='skip', low_memory=False)
            elif ext == '.jsonl': df = pd.read_json(file_path, lines=True)
            elif ext == '.txt':
                df = None
                try:
                    probe = pd.read_csv(file_path, sep='\t', nrows=5, on_bad_lines='skip', engine='python')
                    if probe.shape[1] > 1:
                        df = pd.read_csv(file_path, sep='\t', on_bad_lines='skip', low_memory=False, engine='python')
                except: pass
                if df is None:
                    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f: content = f.read()
                    paragraphs = [p.strip() for p in re.split(r'\n\s*\n', content) if p.strip()]
                    units = paragraphs if len(paragraphs) >= 2 else [l.strip() for l in content.split('\n') if l.strip()]
                    df = pd.DataFrame({'text': units})
            else: continue

            if df is None or df.empty: continue

            text_col = find_text_column(df)
            if not text_col: continue

            df['processed'] = df[text_col].fillna("").astype(str).apply(process)

            cleaned_texts, removed_sentences = strip_generic_boilerplate_per_file(df['processed'].tolist())
            df['processed'] = cleaned_texts

            df['alpha_ratio'] = df['processed'].apply(alpha_ratio_fast)
            df = df[df['alpha_ratio'] > 0.5]
            df = df[df['processed'].str.len() >= 100]

            df['detected_lang'] = df['processed'].apply(detect_lang)
            df = df[df['detected_lang'].isin(['uk', 'en'])]

            for _, row in df.iterrows():
                text = row['processed']
                th = hashlib.md5(text.encode('utf-8')).hexdigest()
                if th in global_exact_hashes: continue

                m = get_minhash(text)
                result = global_lsh.query(m)
                if not result:
                    global_exact_hashes[th] = True
                    key = f"{file_path.name}_{global_id_counter}"
                    global_id_counter += 1
                    global_lsh.insert(key, m)

                    row_dict = {str(k): str(v) for k, v in row.items()}
                    row_dict['source_file'] = str(file_path)

                    out_jsonl.write(json.dumps(row_dict, ensure_ascii=False) + '\n')

        except Exception as e:
            print(f"Error processing file {file_path.name}: {e}")

print(f"Unique items saved: {global_id_counter}")
print("Converting into Parquet...")

try:
    con.execute(f"COPY (SELECT * FROM read_json_auto('{local_jsonl}')) TO '{local_parquet}' (FORMAT PARQUET);")
    print("Copying large files to Google Drive...")

    shutil.copy(local_jsonl, drive_jsonl)
    shutil.copy(local_parquet, drive_parquet)

    print(f"Data saved successfully on Google Drive:\n- {drive_parquet}\n- {drive_jsonl}")
except Exception as e:
    print(f"Error during final conversion or copying: {e}")

Processing...


Processing files:   0%|          | 0/138 [00:00<?, ?it/s]

Unique items saved: 103637
Converting into Parquet...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Copying large files to Google Drive...
Data saved successfully on Google Drive:
- drive/MyDrive/clean_propaganda_dataset.parquet
- drive/MyDrive/clean_propaganda_dataset.jsonl


In [ ]:
df1 = pd.read_json('drive/MyDrive/clean_propaganda_dataset.jsonl', lines=True)
df2 = pd.read_csv('drive/MyDrive/narrative_subnarrative_coding_en.csv')

df_merged = df1.merge(df2, on=['narrative_id', 'sub_narrative_id'], how='left')

In [ ]:
df_merged.to_csv('drive/MyDrive/merged_dataset.csv', index=False)

# 2. Calculating BPC

In [ ]:
pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 21.5 MB/s eta 0:00:00


In [ ]:
import re
import pandas as pd
import uuid
from google.colab import userdata
from huggingface_hub import login
import os
import gc
import json
import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from tqdm import tqdm

In [ ]:
def explode_into_sentences(df, text_column='processed'):
    new_rows = []

    for idx, row in df.iterrows():
        original_text = str(row[text_column])
        doc_id = str(uuid.uuid4())
        sentences = re.split(r'[.!;?]+', original_text)

        for sentence in sentences:
            sentence = sentence.strip()

            if 120 <= len(sentence) <= 200:
                new_row = row.copy()
                new_row['document_id'] = doc_id
                new_row['original_text'] = original_text
                new_row[text_column] = sentence

                new_rows.append(new_row)

    return pd.DataFrame(new_rows)

In [ ]:
df = pd.read_csv('drive/MyDrive/merged_dataset.csv')

df_sentences = explode_into_sentences(df, text_column='processed')


/tmp/ipykernel_598/998680548.py:1: DtypeWarning: Columns (0,1,2,3,4,5,168,169,170,173,174,177,178,179,183,184,185,186,187,189,190,191,192,193,194,196,197) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('drive/MyDrive/merged_dataset.csv')


In [ ]:
df_sentences = pd.read_csv("drive/MyDrive/merged_dataset_split.csv")

/tmp/ipykernel_6653/1024773925.py:1: DtypeWarning: Columns (1,2,3,4,5,6,168,169,170,171,174,175,178,179,180,184,185,186,187,188,190,191,192,193,194,195,197,198) have mixed types. Specify dtype option on import or set low_memory=False.
  df_sentences = pd.read_csv("drive/MyDrive/merged_dataset_split.csv")


In [ ]:
huggingface_token = userdata.get('HF_TOKEN')
login(token=huggingface_token)

In [ ]:
DEVICE = "cuda"

MODELS = [
    "google/gemma-3-1b-pt",
    "meta-llama/Llama-3.2-3B",
    "Qwen/Qwen2.5-7B-Instruct",
    "google/gemma-3-12b-pt",
    "lapa-llm/lapa-12b-pt"
]

USE_4BIT = {
    "Qwen/Qwen2.5-7B-Instruct",
    "google/gemma-3-12b-pt",
    "lapa-llm/lapa-12b-pt"
}

def load_model_and_tokenizer(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name, token=huggingface_token)

    quant_config = None
    if model_name in USE_4BIT:
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_quant_type="nf4",
        )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        quantization_config=quant_config,
        token=huggingface_token
    )
    model.eval()
    return model, tokenizer

def calculate_metrics_per_row(model, tokenizer, df, text_column='text'):
    results = []
    ln2 = float(np.log(2))

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        text = str(row[text_column])

        if len(text) < 120 or len(text) > 200:
            continue

        encoding = tokenizer(text, return_offsets_mapping=True, add_special_tokens=False)
        tokens = encoding['input_ids']
        offsets = encoding['offset_mapping']

        if len(tokens) < 2:
            continue

        bos_id = tokenizer.bos_token_id
        if bos_id is not None:
            tokens = [bos_id] + tokens
            offsets = [(0, 0)] + offsets

        input_ids = torch.tensor([tokens]).to(DEVICE)
        labels = torch.tensor([tokens]).to(DEVICE)

        for i, (start, end) in enumerate(offsets):
            if start < 70:
                labels[0, i] = -100

        with torch.no_grad():
            outputs = model(input_ids=input_ids, labels=labels)
            valid_tokens_count = (labels[0] != -100).sum().item()

            if valid_tokens_count == 0:
                continue

            loss_sum_nats = outputs.loss.item() * valid_tokens_count

        total_bits = loss_sum_nats / ln2
        valid_chars = len(text) - 70

        res_row = row.to_dict()
        res_row['total_bits'] = total_bits
        res_row['valid_chars'] = valid_chars
        res_row['bpc'] = total_bits / valid_chars if valid_chars > 0 else float('nan')

        results.append(res_row)

    return pd.DataFrame(results)

def run_pipeline(df, output_dir="./results"):
    os.makedirs(output_dir, exist_ok=True)

    for model_name in MODELS:
        print(f"\nЗапуск моделі: {model_name}")
        model, tokenizer = load_model_and_tokenizer(model_name)

        df_results = calculate_metrics_per_row(model, tokenizer, df)
        df_results['model_name'] = model_name

        safe_model_name = model_name.replace("/", "_")
        output_path = os.path.join(output_dir, f"entropy_{safe_model_name}.csv")

        df_results.to_csv(output_path, index=False)
        print(f"Результати збережено у файл: {output_path}")

        del model, tokenizer
        torch.cuda.empty_cache()
        gc.collect()

In [ ]:
run_pipeline(df_sentences, output_dir="/content/drive/MyDrive/results")


Запуск моделі: google/gemma-3-1b-pt


config.json:   0%|          | 0.00/880 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

 20%|█▉        | 20325/104192 [25:29<1:30:23, 15.46it/s]